- **2. 生成 dataset.json**：编写 Python 脚本，将 `step_4_metadata.csv` 转换为 JSON。路径全部指向**原始的 `.nii.gz` 文件**（不再指向任何 Latent 缓存）。
    - 映射关系：`image` -> Sub；`condition` -> Pre；`label` -> Mask。

In [7]:
import pandas as pd
import json
import os
from sklearn.model_selection import train_test_split

# --- 配置区 ---
CSV_PATH = "/mnt/ext_drive/breast/step_4/step_4_metadata.csv"
SAVE_PATH = "breast_project/data/dataset.json"

def get_source_name(original_id):
    """从 Original_ID 中提取数据集来源"""
    orig_id = str(original_id).upper()
    if 'DUKE' in orig_id: return 'DUKE'
    if 'ISPY1' in orig_id: return 'ISPY1'
    if 'ISPY2' in orig_id: return 'ISPY2'
    if 'NACT' in orig_id: return 'NACT'
    return 'OTHER'

def generate_maisi_json():
    # 1. 加载数据
    df = pd.read_csv(CSV_PATH)
    
    # 2. 提取数据源
    df['Source'] = df['Original_ID'].apply(get_source_name)
    
    print("\n" + "="*40)
    print("📊 你的全量数据分布 (按来源与有无肿瘤):")
    print("="*40)
    distribution = df.groupby(['Source', 'Has_Tumor']).size().unstack(fill_value=0)
    distribution['Total'] = distribution.sum(axis=1)
    print(distribution)
    print(f"\n总数据量: {len(df)}")
    print("="*40 + "\n")

    # ------------------ Patient-Level Split (终极修复版) ------------------
    # 3. 按患者 (Original_ID) 进行聚合
    patient_df = df.groupby('Original_ID').agg({
        'Source': 'first',
        'Has_Tumor': 'max'
    }).reset_index()
    
    patient_df['Patient_Stratify_Key'] = patient_df['Source'] + "_Tumor_" + patient_df['Has_Tumor'].astype(str)
    
    # 识别出能正常分层的患者和极少数患者（孤本）
    class_counts = patient_df['Patient_Stratify_Key'].value_counts()
    valid_classes = class_counts[class_counts >= 2].index
    
    patient_df_stratifiable = patient_df[patient_df['Patient_Stratify_Key'].isin(valid_classes)]
    patient_df_rare = patient_df[~patient_df['Patient_Stratify_Key'].isin(valid_classes)]
    
    # 计算验证集抽取比例 (目标是抽出约 100 个有效切块样本)
    val_ratio = 100 / len(df) 
    
    # 仅对充足的患者群体进行完美分层抽样
    train_patients_strat, val_patients = train_test_split(
        patient_df_stratifiable, 
        test_size=val_ratio, 
        stratify=patient_df_stratifiable['Patient_Stratify_Key'], 
        random_state=42
    )

    # 将极少数患者强制归入训练集（模型必须学到这些罕见特征）
    train_patients = pd.concat([train_patients_strat, patient_df_rare])

    # 把具体的乳腺切块样本分发到训练集和验证集
    train_df = df[df['Original_ID'].isin(train_patients['Original_ID'])]
    val_df = df[df['Original_ID'].isin(val_patients['Original_ID'])]
    # ------------------ 修改结束 ------------------

    # --- [验证集质量检查] ---
    print(f"🧪 严格隔离患者后，抽取出的验证集分布情况 (共 {len(val_df)} 个样本):")
    val_dist = val_df.groupby(['Source', 'Has_Tumor']).size().unstack(fill_value=0)
    print(val_dist)
    print("="*40 + "\n")

    # 4. 构建 MAISI JSON 格式
    dataset = {
        "description": "Breast Subtraction Generation Dataset (Stratified)",
        "labels": {"0": "background", "1": "tumor"},
        "tensorImageSize": "3D",
        "modality": { "0": "mri", "1": "ct" }, # 兼容 MAISI 强制依赖
        "training": [],
        "validation": []
    }

    def row_to_dict(row):
        return {
            "image": row['Path_Sub'],      
            "condition": row['Path_Pre'],  
            "label": row['Path_Mask'],     
            "modality": "mri",             
            "has_tumor": int(row['Has_Tumor']),
            "uuid": row['UUID']
        }

    dataset["training"] = train_df.apply(row_to_dict, axis=1).tolist()
    dataset["validation"] = val_df.apply(row_to_dict, axis=1).tolist()

    # 5. 保存
    os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)
    with open(SAVE_PATH, 'w') as f:
        json.dump(dataset, f, indent=4)
    
    print(f"✅ 成功！高质量平衡 JSON 已保存至: {SAVE_PATH}")
    print(f"📈 最终训练集: {len(dataset['training'])} | 验证集: {len(dataset['validation'])}")

if __name__ == "__main__":
    generate_maisi_json()


📊 你的全量数据分布 (按来源与有无肿瘤):
Has_Tumor    0    1  Total
Source                    
DUKE       283  289    572
ISPY1        4  171    175
ISPY2      153  979   1132
NACT         0   64     64

总数据量: 1943

🧪 严格隔离患者后，抽取出的验证集分布情况 (共 103 个样本):
Has_Tumor   0   1
Source           
DUKE       15  15
ISPY1       1   9
ISPY2       9  51
NACT        0   3

✅ 成功！高质量平衡 JSON 已保存至: breast_project/data/dataset.json
📈 最终训练集: 1840 | 验证集: 103
